# Indra OCR — 1M-parameter CPU document OCR

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/Indra-OCR-1M-document-ocr.ipynb)

[`datalab-to/indra-ocr`](https://huggingface.co/datalab-to/indra-ocr) is Datalab's ~1M-parameter OCR model. It runs on **CPU** with ONNX Runtime (no PyTorch, no GPU) and turns PDFs and images into markdown, plain text, or JSON with line and layout boxes. It reads handwriting, 80+ languages, and pages that are rotated or tilted. The card reports **66.7** on [OmniParseBench](https://github.com/datalab-to/omni_parse_bench) (**71.4** with `--math`), ahead of PaddleOCR PP-StructureV3 (58.4) and Tesseract's text score (19.3), and about **210 ms/page on 4 CPU cores**.

Default models are about **76 MB** (30 MB of networks + 46 MB of language models for unsure lines). The main Latin recognizer `rec.onnx` is about 1.25M parameters; all default models together are 7.3M. Optional math models download only when `--math` / `math=True` is on.

This notebook follows the official install and API from the [model card](https://huggingface.co/datalab-to/indra-ocr) and [`datalab-to/indra`](https://github.com/datalab-to/indra): `pip install indra-ocr`, then `indra_ocr.load` / the `indra-ocr` CLI.

**Licence:** Code [Apache 2.0](https://github.com/datalab-to/indra/blob/main/LICENSE). Model weights [OpenRAIL-M](https://huggingface.co/datalab-to/indra-ocr/blob/main/LICENSE) (modified: free for research, personal use, and startups under $20M funding/revenue; cannot be used competitively with Datalab's API).

**Runtime:** *Runtime > Change runtime type > CPU* is enough. A **T4** runtime also works — Indra OCR stays on ONNX Runtime CPU and does not use the GPU. The first run downloads about 76 MB from Hugging Face.

**Not in this notebook:** `indra-ocr-viewer` (a local web UI on `localhost:8765`). Run that on your machine, not in Colab.

**Links:** [model card](https://huggingface.co/datalab-to/indra-ocr) · [GitHub](https://github.com/datalab-to/indra) · [docs](https://documentation.datalab.to) · [playground](https://www.datalab.to/playground) · [PyPI](https://pypi.org/project/indra-ocr/) · [official quickstart](https://github.com/datalab-to/indra/blob/main/examples/quickstart.ipynb)


## 1. Install

Official install is `pip install indra-ocr`. It pulls ONNX Runtime, OpenCV, pypdfium2, and huggingface_hub — not PyTorch. `1.1.2` is the current PyPI release.


In [ ]:
%pip install -q "indra-ocr==1.1.2" pillow matplotlib requests


## 2. Load Indra OCR

`indra_ocr.load(threads=4)` is the card's recipe. `threads` is ONNX Runtime intra-op threads for this process. The CLI / `convert_many` instead start one single-threaded worker per core; in a notebook we keep one converter and call `convert` / `ocr` on it.

Useful `load` kwargs (all from the card):

* `mode="auto"` — read a PDF text layer when it passes the checks, OCR elsewhere (`"ocr"` never reads the layer; `"text"` always does)
* `math=True` — write math as inline LaTeX (`$...$`); off by default, slower on math pages
* `furniture=True` — keep page headers and footers in the markdown (`--keep-furniture` on the CLI)


In [ ]:
import os

import indra_ocr

THREADS = min(4, os.cpu_count() or 1)
conv = indra_ocr.load(threads=THREADS)

print("indra-ocr", indra_ocr.__version__)
print("threads", THREADS, "| cores", os.cpu_count())
print("model dir", indra_ocr.model_dir())


## 3. Sample pages

Same fetch style as the [LightOnOCR notebook](LightOnOCR-3-0.8B-document-ocr.ipynb): retry on 429/5xx, and prefer a repo mirror when Wikimedia rate-limits Colab IPs.

| Sample | Why | Source |
| --- | --- | --- |
| SROIE receipt | printed page / receipt | Hugging Face OCR fixtures (same file LightOnOCR uses) |
| Attention Is All You Need, pages 1–3 | multi-page digital PDF | [arXiv:1706.03762](https://arxiv.org/pdf/1706.03762) |
| Handwritten resignation letter | handwriting | Wikimedia Commons, [CC BY 2.0](https://creativecommons.org/licenses/by/2.0/), mirrored in this repo |
| Official `scan.png` | non-Latin (Korean letter) | [`datalab-to/indra` examples](https://github.com/datalab-to/indra/tree/main/examples/samples) (ODC-BY 1.0) |
| Official `equations.png` | `--math` / LaTeX | same official samples folder (ODC-BY 1.0) |
| Receipt rotated 90° | orientation model | built here from the SROIE image |

Indra's own card lists handwriting as a weak spot (25.2% on OmniParseBench handwriting tests). The letter cell will still run; expect errors on cursive.


In [ ]:
import io
import time
from pathlib import Path

import numpy as np
import requests
from IPython.display import display
from PIL import Image, ImageDraw, ImageFont

SAMPLE_DIR = Path("indra_samples")
SAMPLE_DIR.mkdir(exist_ok=True)

RECEIPT_URL = "https://huggingface.co/datasets/hf-internal-testing/fixtures_ocr/resolve/main/SROIE-receipt.jpeg"
PAPER_URL = "https://arxiv.org/pdf/1706.03762"
INDRA_SAMPLES = "https://raw.githubusercontent.com/datalab-to/indra/main/examples/samples/"
INDRA_SAMPLES_FALLBACK = "https://cdn.jsdelivr.net/gh/datalab-to/indra@main/examples/samples/"
LETTER_URLS = [
    "https://raw.githubusercontent.com/donvito/notebooks/main/colab/data/lightonocr/handwritten-letter-of-resignation.jpg",
    "https://upload.wikimedia.org/wikipedia/commons/e/eb/Handwritten_Letter_of_Resignation.jpg",
]
UA = {"User-Agent": "donvito-notebooks/1.0 (educational Colab demo)"}


def fetch_bytes(url, attempts=4):
    for attempt in range(attempts):
        response = requests.get(url, headers=UA, timeout=120)
        if response.status_code not in (429, 500, 502, 503, 504) or attempt == attempts - 1:
            response.raise_for_status()
            return response.content
        retry_after = response.headers.get("Retry-After", "")
        time.sleep(int(retry_after) if retry_after.isdigit() else 2 ** (attempt + 1))


def fetch_to(path, *urls):
    path = Path(path)
    if path.exists() and path.stat().st_size > 0:
        return path
    last_error = None
    for url in urls:
        try:
            path.write_bytes(fetch_bytes(url))
            print("saved", path.name, "from", url.split("/")[-1], f"({path.stat().st_size} bytes)")
            return path
        except requests.RequestException as error:
            last_error = error
            print(f"Could not fetch {url}: {error}")
    raise RuntimeError(f"All URLs failed for {path.name}: {last_error}")


def load_image(path, longest=1600):
    image = Image.open(path).convert("RGB")
    image.thumbnail((longest, longest))
    return image


def show(image, title=None, longest=720):
    preview = image.copy()
    preview.thumbnail((longest, longest))
    if title:
        print(title, preview.size)
    display(preview)


receipt_path = fetch_to(SAMPLE_DIR / "receipt.jpg", RECEIPT_URL)
scan_path = fetch_to(SAMPLE_DIR / "scan.png", INDRA_SAMPLES + "scan.png", INDRA_SAMPLES_FALLBACK + "scan.png")
equations_path = fetch_to(
    SAMPLE_DIR / "equations.png",
    INDRA_SAMPLES + "equations.png",
    INDRA_SAMPLES_FALLBACK + "equations.png",
)
letter_path = fetch_to(SAMPLE_DIR / "letter.jpg", *LETTER_URLS)
paper_path = fetch_to(SAMPLE_DIR / "attention-is-all-you-need.pdf", PAPER_URL)

receipt = load_image(receipt_path)
scan = load_image(scan_path)
equations = load_image(equations_path)
letter = load_image(letter_path)
rotated = receipt.rotate(90, expand=True)
rotated_path = SAMPLE_DIR / "receipt-rotated-90.jpg"
rotated.save(rotated_path, quality=92)

print("receipt", receipt.size, "| scan", scan.size, "| letter", letter.size)
print("equations", equations.size, "| rotated", rotated.size)
print("pdf", paper_path.stat().st_size, "bytes")


## 4. Printed receipt — markdown and the CLI

`convert` takes a file path (image or PDF) and yields `(page_index, Page)`. `Page.markdown` is headings, lists, and tables as HTML. `Page.text` is one line per detected line.

The CLI is the same pipeline: `indra-ocr file --format md|text|json`, plus `--mode`, `--math`, `--keep-furniture`, `--timings`, `--cores`, `--dpi`, `--max-pages`.


In [ ]:
import subprocess
from dataclasses import asdict

show(receipt, "SROIE receipt")

page_receipt = None
for i, page in conv.convert(receipt_path):
    page_receipt = page
    print(f"page {i} source={page.source} rotation={page.rotation} skew={page.skew:.1f}")
    print(f"lines={len(page.lines)} blocks={len(page.blocks)} total_ms={page.timings.get('total_ms', 0):.0f}")
    print("--- markdown ---")
    print(page.markdown or page.text)

print("\n--- CLI: indra-ocr --format text --timings ---")
cli = subprocess.run(
    ["indra-ocr", str(receipt_path), "--format", "text", "--timings", "--cores", "1"],
    check=True,
    capture_output=True,
    text=True,
)
print(cli.stdout)
if cli.stderr:
    print(cli.stderr.strip())


## 5. JSON with boxes drawn on the page

`--format json` / the `Page` object give every line (`box`, `text`, `conf`) and every layout block (`label`, `bbox`, `text`). Boxes are **image pixels**. The official examples draw faint blue line boxes and coloured layout blocks; this cell does the same with PIL and shows the overlay with matplotlib.


In [ ]:
import json

import matplotlib.pyplot as plt

BLOCK_COLOURS = {
    "Text": (40, 110, 210),
    "Section-Header": (200, 40, 40),
    "Page-Header": (120, 120, 120),
    "Page-Footer": (120, 120, 120),
    "List-Group": (20, 140, 80),
    "Table": (180, 90, 20),
    "Caption": (140, 60, 180),
    "Footnote": (90, 90, 90),
    "Table-Of-Contents": (20, 140, 140),
    "Form": (200, 140, 20),
}


def page_json(page, file_name, index=0):
    return {
        "file": file_name,
        "page": index,
        "source": page.source,
        "rotation": page.rotation,
        "skew": page.skew,
        "vertical": page.vertical,
        "note": page.note,
        "lines": [asdict(ln) for ln in page.lines],
        "blocks": page.blocks,
        "timings": page.timings,
        "markdown": page.markdown,
    }


def draw_boxes(image, page):
    canvas = image.copy()
    draw = ImageDraw.Draw(canvas)
    try:
        font = ImageFont.load_default()
    except OSError:
        font = None
    for ln in page.lines:
        x0, y0, x1, y1 = ln.box
        draw.rectangle([x0, y0, x1, y1], outline=(90, 160, 230), width=2)
    for block in page.blocks:
        x0, y0, x1, y1 = block["bbox"]
        colour = BLOCK_COLOURS.get(block["label"], (220, 40, 40))
        draw.rectangle([x0, y0, x1, y1], outline=colour, width=3)
        draw.text((x0 + 3, max(y0 - 12, 0)), block["label"], fill=colour, font=font)
    return canvas


payload = page_json(page_receipt, receipt_path.name)
print(json.dumps({k: payload[k] for k in ("source", "rotation", "skew", "timings")}, indent=2))
print("first 4 lines:")
print(json.dumps(payload["lines"][:4], indent=2, ensure_ascii=False))
print("blocks:")
print(json.dumps([{"label": b["label"], "bbox": b["bbox"], "text": b["text"][:80]} for b in payload["blocks"]], indent=2, ensure_ascii=False))

boxed = draw_boxes(receipt, page_receipt)
fig, axes = plt.subplots(1, 2, figsize=(10, 6))
axes[0].imshow(receipt)
axes[0].set_title("receipt")
axes[1].imshow(boxed)
axes[1].set_title("JSON boxes (blue = lines, colour = layout)")
for ax in axes:
    ax.set_axis_off()
fig.tight_layout()
plt.show()


## 6. Multi-page PDF

`convert(path, max_pages=3)` reads the first three pages of [Attention Is All You Need](https://arxiv.org/pdf/1706.03762). Default `mode="auto"` uses the PDF text layer when it is trustworthy (`page.source` is `text` or `hybrid`). `--mode ocr` / a second converter with `mode="ocr"` ignores the layer and OCRs the render — useful for scans, or to compare the two paths.


In [ ]:
print("=== auto (text layer when trusted) ===")
pdf_pages = []
for i, page in conv.convert(paper_path, max_pages=3):
    pdf_pages.append(page)
    excerpt = (page.markdown or page.text or "").strip().splitlines()
    print(f"page {i + 1} source={page.source} lines={len(page.lines)} "
          f"blocks={len(page.blocks)} total_ms={page.timings.get('total_ms', 0):.0f}")
    print("\n".join(excerpt[:8]))
    print("...")

print("\n=== mode=ocr on page 1 (ignore the text layer) ===")
conv_ocr = indra_ocr.load(threads=THREADS, mode="ocr")
_, page_ocr = next(conv_ocr.convert(paper_path, max_pages=1))
print(f"source={page_ocr.source} lines={len(page_ocr.lines)} total_ms={page_ocr.timings.get('total_ms', 0):.0f}")
print((page_ocr.markdown or page_ocr.text)[:600])


## 7. Handwriting

The card is honest here: handwriting reads, but not well (about a third of the words on held-out pages; cursive and faint pencil are worst). This is the same CIPHR Connect letter the LightOnOCR notebook uses.


In [ ]:
show(letter, "handwritten letter")
_, page_letter = next(conv.convert(letter_path))
print(f"source={page_letter.source} lines={len(page_letter.lines)} "
      f"total_ms={page_letter.timings.get('total_ms', 0):.0f} note={page_letter.note}")
print("--- markdown ---")
print(page_letter.markdown or page_letter.text)
show(draw_boxes(letter, page_letter), "handwriting boxes")


## 8. Non-Latin script — Korean letter

Official `examples/samples/scan.png`: a scanned Korean letter (OmniParseBench / FinePDFs, ODC-BY 1.0). A line classifier sends each line to `rec.onnx` (Latin) or `rec_cjk.onnx` (Chinese, Japanese, Korean). Arabic/Hebrew go to `rec_rtl.onnx`; Indic, Thai, Khmer and others to `rec_world.onnx`.


In [ ]:
show(scan, "Korean letter (official scan.png)")
_, page_scan = next(conv.convert(scan_path))
print(f"source={page_scan.source} lines={len(page_scan.lines)} "
      f"total_ms={page_scan.timings.get('total_ms', 0):.0f}")
print("--- markdown ---")
print(page_scan.markdown or page_scan.text)
show(draw_boxes(scan, page_scan), "Korean boxes")


## 9. Rotated page

`orient.onnx` turns a page that is a quarter or half turn upright (`page.rotation` is the counter-clockwise turn it applied). A small tilt up to about 45° is straightened (`page.skew`); boxes are mapped back onto the page as you handed it in. Here the SROIE receipt is rotated 90° with PIL.


In [ ]:
show(rotated, "receipt rotated 90° CCW")
_, page_rot = next(conv.convert(rotated_path))
print(f"source={page_rot.source} rotation={page_rot.rotation} skew={page_rot.skew:.1f} "
      f"lines={len(page_rot.lines)} total_ms={page_rot.timings.get('total_ms', 0):.0f}")
print("--- markdown ---")
print(page_rot.markdown or page_rot.text)
show(draw_boxes(rotated, page_rot), "boxes on the rotated page")


## 10. Math mode (`--math` / `math=True`)

Off by default. With it, a small classifier finds lines that hold math and a second recognizer writes them as text with inline LaTeX (`we have $x _ { i } \leq 1$`). That is the 66.7 → 71.4 OmniParseBench jump. The math models download on first use. Official `equations.png` is an arXiv page from olmOCR-bench.


In [ ]:
show(equations, "equations.png")

print("=== default (math off) ===")
_, page_eq = next(conv.convert(equations_path))
print(page_eq.markdown or page_eq.text)

print("\n=== math=True ===")
conv_math = indra_ocr.load(threads=THREADS, math=True)
_, page_eq_math = next(conv_math.convert(equations_path))
print(f"total_ms default={page_eq.timings.get('total_ms', 0):.0f} "
      f"math={page_eq_math.timings.get('total_ms', 0):.0f}")
print(page_eq_math.markdown or page_eq_math.text)


## 11. Timing — pages/sec

The card quotes ~210 ms/page on 4 cores (one page at a time) and ~6.2 pages/s with one worker per core on 4 cores. Colab free CPU is usually 2 cores and a noisy neighbour, so this is a ballpark, not a leaderboard run. Times below are `page.timings["total_ms"]` after the model is already loaded (the first page of a process includes warmup).


In [ ]:
timed = [
    ("receipt", page_receipt),
    ("pdf p1 auto", pdf_pages[0]),
    ("pdf p1 ocr", page_ocr),
    ("handwriting", page_letter),
    ("Korean scan", page_scan),
    ("rotated receipt", page_rot),
    ("equations", page_eq),
    ("equations --math", page_eq_math),
]

print(f"{'sample':<22} {'ms':>8} {'pages/s':>8}  source")
total_ms = 0.0
for name, page in timed:
    ms = float(page.timings.get("total_ms", 0.0))
    total_ms += ms
    pps = (1000.0 / ms) if ms else float("inf")
    print(f"{name:<22} {ms:8.0f} {pps:8.2f}  {page.source}")

print(f"\n{len(timed)} pages in {total_ms:.0f} ms → {1000.0 * len(timed) / max(total_ms, 1):.2f} pages/s (serial, this runtime)")
print("per-step timings on the receipt:", {k: round(v, 1) for k, v in page_receipt.timings.items()})


## 12. Optional: one-page side-by-side vs Tesseract

The card compares Indra's text score (46.7) with Tesseract 5 Latin (19.3) on OmniParseBench. This cell installs the Tesseract binary when it is missing, then runs `pytesseract` on the same receipt. Skip it if `apt-get` is not available.


In [ ]:
import shutil
import sys

def ensure_tesseract():
    if shutil.which("tesseract"):
        return True
    if not shutil.which("apt-get"):
        print("No tesseract binary and no apt-get; skip the comparison.")
        return False
    print("Installing tesseract-ocr ...")
    update = subprocess.run(["apt-get", "-qq", "update"], check=False)
    install = subprocess.run(
        ["apt-get", "-qq", "install", "-y", "tesseract-ocr"],
        check=False,
        capture_output=True,
        text=True,
    )
    if install.returncode != 0:
        print(install.stderr[-800:] if install.stderr else "apt-get install failed")
        return False
    return shutil.which("tesseract") is not None


if ensure_tesseract():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytesseract"], check=True)
    import pytesseract

    t0 = time.perf_counter()
    tess = pytesseract.image_to_string(receipt)
    tess_ms = (time.perf_counter() - t0) * 1000
    print(f"Tesseract {tess_ms:.0f} ms | Indra {page_receipt.timings.get('total_ms', 0):.0f} ms")
    print("--- Tesseract ---")
    print(tess.strip() or "(empty)")
    print("--- Indra text ---")
    print(page_receipt.text)
else:
    print("Skipped.")
